# 1. Import the necessary libraries

In [ ]:
# Adds the parent directory to sys.path
import sys
sys.path.append('../')

from problems.benchmark_problems import get_pareto

from autorank import autorank
from pymoo.indicators.hv import HV
from pymoo.indicators.igd import IGD

import numpy as np
import pandas as pd
import pickle
import statistics

results_folder = '../results/'

# 2. Hypervolume

## 2.1 Hypervolume DataFrame

In [ ]:
#################### CUSTOMIZABLE ####################

# Experiment configuration
experiment_names = ['dtlz1', 'dtlz2']
objective_dim = 2
position_dim = 10

# Consider other results
insert_cmopso = True
inset_maco = True
insert_mopso_cd = True
insert_nsga2 = True
insert_nspso = True
insert_spea2 = True
######################################################

# Create the hypervolume indicator
ref_point = np.array([10] * objective_dim)
indicator = HV(ref_point=ref_point)

In [ ]:
# List of file tuples
file_tuple_cmopso = []
file_tuple_maco = []
file_tuple_mopso_cd = []
file_tuple_nsga2 = []
file_tuple_nspso = []
file_tuple_spea2 = []

# Algorithm's file tuple
file_tuple_mesh = [
        (f"MESH_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH', exp_name)
        for exp_name in experiment_names
    ]

if insert_cmopso:
    file_tuple_cmopso = [(f'CMOPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'CMOPSO', exp_name) for exp_name in experiment_names]
if inset_maco:
    file_tuple_maco = [(f'MACO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MACO', exp_name) for exp_name in experiment_names]
if insert_mopso_cd:
    file_tuple_mopso_cd = [(f'MOPSO_CD_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD', exp_name) for exp_name in experiment_names]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', exp_name) for exp_name in experiment_names]
if insert_nspso:
    file_tuple_nspso = [(f'NSPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO', exp_name) for exp_name in experiment_names]
if insert_spea2:
    file_tuple_spea2 = [(f'SPEA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'SPEA2', exp_name) for exp_name in experiment_names]


file_tuple = file_tuple_mesh + file_tuple_cmopso + file_tuple_maco + file_tuple_mopso_cd + file_tuple_nsga2 + file_tuple_nspso + file_tuple_spea2

# Take the results
results_mesh_hv = []
results_cmopso_hv = []
results_maco_hv = []
results_mopso_cd_hv = []
results_nsga2_hv = []
results_nspso_hv = []
results_spea2_hv = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open(results_folder + file_name, "rb") as f:
		results_mesh_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_cmopso:
    with open(results_folder + file_name, "rb") as f:
        results_cmopso_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_maco:
    with open(results_folder + file_name, "rb") as f:
        results_maco_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_mopso_cd:
    with open(results_folder + file_name, "rb") as f:
        results_mopso_cd_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open(results_folder + file_name, "rb") as f:
		results_nsga2_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_nspso:
    with open(results_folder + file_name, "rb") as f:
        results_nspso_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_spea2:
    with open(results_folder + file_name, "rb") as f:
        results_spea2_hv.append(pickle.load(f))

results = results_mesh_hv + results_cmopso_hv + results_maco_hv + results_mopso_cd_hv + results_nsga2_hv + results_nspso_hv + results_spea2_hv

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean HV', 'Std. Dev.', 'Median', 'Combined HV', 'Min. HV', 'Max. HV']

# Store the datas
data = [[
		alg_name,
		func_name.upper(),
		statistics.mean(HVs := [indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)]),
		statistics.stdev(HVs) if len(HVs) > 1 else 0,
		statistics.median(HVs) if len(HVs) > 1 else HVs[0],
		indicator(results[i]['combined'][1]),
		min(HVs),
		max(HVs),
		]
		for i, (_, alg_name, func_name) in enumerate(file_tuple)
		for r in [results[i]]
	]

# Create the DataFrame with hypervolume results
df_hv = pd.DataFrame(data, columns=columns)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(df_hv.sort_values(by=['Median'], ascending=False))

In [ ]:
print(df_hv.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Hypervolume indicator of the algorithms on the test problems.'))

# 3. Inverse Generational Distance

## 3.1 Inverse Generational Distance DataFrame

In [ ]:
#################### CUSTOMIZABLE ####################

# Experiment configuration
experiment_name = 'dtlz1'
objective_dim = 2
position_dim = 10
wfg_k = 6
n_pareto_density = 100

# Consider other results
insert_cmopso = True
inset_maco = True
insert_mopso_cd = True
insert_nsga2 = True
insert_nspso = True
insert_spea2 = True
######################################################

# Create the IGD indicator
indicator = IGD(get_pareto(experiment_name, n_pareto_density, position_dim, objective_dim, wfg_k))

In [ ]:
# List of file tuples
file_tuple_cmopso = []
file_tuple_maco = []
file_tuple_mopso_cd = []
file_tuple_nsga2 = []
file_tuple_nspso = []
file_tuple_spea2 = []

# Algorithm's file tuple
file_tuple_mesh = [
        (f"MESH_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH', exp_name)
        for exp_name in experiment_names
    ]

if insert_cmopso:
    file_tuple_cmopso = [(f'CMOPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'CMOPSO', exp_name) for exp_name in experiment_names]
if inset_maco:
    file_tuple_maco = [(f'MACO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MACO', exp_name) for exp_name in experiment_names]
if insert_mopso_cd:
    file_tuple_mopso_cd = [(f'MOPSO_CD_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD', exp_name) for exp_name in experiment_names]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', exp_name) for exp_name in experiment_names]
if insert_nspso:
    file_tuple_nspso = [(f'NSPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO', exp_name) for exp_name in experiment_names]
if insert_spea2:
    file_tuple_spea2 = [(f'SPEA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'SPEA2', exp_name) for exp_name in experiment_names]


file_tuple = file_tuple_mesh + file_tuple_cmopso + file_tuple_maco + file_tuple_mopso_cd + file_tuple_nsga2 + file_tuple_nspso + file_tuple_spea2

# Take the results
results_mesh_igd = []
results_cmopso_igd = []
results_maco_igd = []
results_mopso_cd_igd = []
results_nsga2_igd = []
results_nspso_igd = []
results_spea2_igd = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open(results_folder + file_name, "rb") as f:
		results_mesh_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_cmopso:
    with open(results_folder + file_name, "rb") as f:
        results_cmopso_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_maco:
    with open(results_folder + file_name, "rb") as f:
        results_maco_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_mopso_cd:
    with open(results_folder + file_name, "rb") as f:
        results_mopso_cd_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open(results_folder + file_name, "rb") as f:
		results_nsga2_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_nspso:
    with open(results_folder + file_name, "rb") as f:
        results_nspso_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_spea2:
    with open(results_folder + file_name, "rb") as f:
        results_spea2_igd.append(pickle.load(f))

results = results_mesh_igd + results_cmopso_igd + results_maco_igd + results_mopso_cd_igd + results_nsga2_igd + results_nspso_igd + results_spea2_igd

# Create the pandas DataFrame columns
columns = ['Algorithm', 'Function', 'Mean IGD', 'Std. Dev.', 'Median', 'Combined IGD', 'Min. IGD', 'Max. IGD']

# Store the datas
data = [[
		alg_name,
		func_name.upper(),
		statistics.mean(IGDs := [indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)]),
		statistics.stdev(IGDs) if len(IGDs) > 1 else 0,
		statistics.median(IGDs) if len(IGDs) > 1 else IGDs[0],
		indicator(results[i]['combined'][1]),
		min(IGDs),
		max(IGDs),
		]
		for i, (_, alg_name, func_name) in enumerate(file_tuple)
		for r in [results[i]]
	]

# Create the DataFrame with hypervolume results
df_igd = pd.DataFrame(data, columns=columns)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(df_igd.sort_values(by=['Median'], ascending=True))

In [ ]:
print(df_igd.to_latex(index=False, float_format="%.3f", escape=False, column_format='lcccccc', label='tab:hv', caption='Inverse Generational Distance indicator of the algorithms on the test problems.'))

# 4. Hypothesis Test

## 4.1 Test for Hypervolume by Autorank

In [ ]:
#################### CUSTOMIZABLE ####################
# Significance level
alpha_hv = 0.05

# Experiment configuration
experiment_name = 'dtlz1'
objective_dim = 2
position_dim = 10


# Consider other results
insert_cmopso = True
inset_maco = True
insert_mopso_cd = True
insert_nsga2 = True
insert_nspso = True
insert_spea2 = True
######################################################

# Create the hypervolume indicator
ref_point = np.array([10] * objective_dim)
indicator = HV(ref_point=ref_point)

In [ ]:
# List of file tuples
file_tuple_cmopso = []
file_tuple_maco = []
file_tuple_mopso_cd = []
file_tuple_nsga2 = []
file_tuple_nspso = []
file_tuple_spea2 = []

# Algorithm's file tuple
file_tuple_mesh = [
        (f"MESH_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH', exp_name)
        for exp_name in experiment_names
    ]

if insert_cmopso:
    file_tuple_cmopso = [(f'CMOPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'CMOPSO', exp_name) for exp_name in experiment_names]
if inset_maco:
    file_tuple_maco = [(f'MACO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MACO', exp_name) for exp_name in experiment_names]
if insert_mopso_cd:
    file_tuple_mopso_cd = [(f'MOPSO_CD_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD', exp_name) for exp_name in experiment_names]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', exp_name) for exp_name in experiment_names]
if insert_nspso:
    file_tuple_nspso = [(f'NSPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO', exp_name) for exp_name in experiment_names]
if insert_spea2:
    file_tuple_spea2 = [(f'SPEA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'SPEA2', exp_name) for exp_name in experiment_names]


file_tuple = file_tuple_mesh + file_tuple_cmopso + file_tuple_maco + file_tuple_mopso_cd + file_tuple_nsga2 + file_tuple_nspso + file_tuple_spea2

# Take the results
results_mesh_hv = []
results_cmopso_hv = []
results_maco_hv = []
results_mopso_cd_hv = []
results_nsga2_hv = []
results_nspso_hv = []
results_spea2_hv = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open(results_folder + file_name, "rb") as f:
		results_mesh_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_cmopso:
    with open(results_folder + file_name, "rb") as f:
        results_cmopso_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_maco:
    with open(results_folder + file_name, "rb") as f:
        results_maco_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_mopso_cd:
    with open(results_folder + file_name, "rb") as f:
        results_mopso_cd_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open(results_folder + file_name, "rb") as f:
		results_nsga2_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_nspso:
    with open(results_folder + file_name, "rb") as f:
        results_nspso_hv.append(pickle.load(f))
for file_name, _, _ in file_tuple_spea2:
    with open(results_folder + file_name, "rb") as f:
        results_spea2_hv.append(pickle.load(f))

results = results_mesh_hv + results_cmopso_hv + results_maco_hv + results_mopso_cd_hv + results_nsga2_hv + results_nspso_hv + results_spea2_hv

# Create the pandas DataFrame columns
columns = [alg_name for (_, alg_name, _) in file_tuple]

hv_values = [[indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)] for i in range(len(results))]

df_hv = pd.DataFrame(list(zip(*hv_values)), columns=columns)
autorank_report_hv = autorank(df_hv, order='descending', alpha=alpha_hv)

In [ ]:
autorank_hv = autorank_report_hv[0]
CD_hv = autorank_report_hv[2]
# If the differences between the mean ranks of two algorithms is greater than the critical distance, they are statistically different
print('Critical Distance (Hypervolume): ', CD_hv)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(autorank_hv.sort_values(by=['meanrank'], ascending=True))

## 4.2 Test for Inverse Generational Distance by Autorank

In [ ]:
#################### CUSTOMIZABLE ####################
# Significance level
alpha_igd = 0.05

# Experiment configuration
experiment_name = 'dtlz1'
objective_dim = 2
position_dim = 10

# Consider other results
insert_cmopso = True
inset_maco = True
insert_mopso_cd = True
insert_nsga2 = True
insert_nspso = True
insert_spea2 = True
######################################################

# Create the IGD indicator
indicator = IGD(get_pareto(experiment_name, n_pareto_density, position_dim, objective_dim, wfg_k))

In [ ]:
# List of file tuples
file_tuple_cmopso = []
file_tuple_maco = []
file_tuple_mopso_cd = []
file_tuple_nsga2 = []
file_tuple_nspso = []
file_tuple_spea2 = []

# Algorithm's file tuple
file_tuple_mesh = [
        (f"MESH_{exp_name}_{objective_dim}_{position_dim}.pkl", 'MESH', exp_name)
        for exp_name in experiment_names
    ]

if insert_cmopso:
    file_tuple_cmopso = [(f'CMOPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'CMOPSO', exp_name) for exp_name in experiment_names]
if inset_maco:
    file_tuple_maco = [(f'MACO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MACO', exp_name) for exp_name in experiment_names]
if insert_mopso_cd:
    file_tuple_mopso_cd = [(f'MOPSO_CD_{exp_name}_{objective_dim}_{position_dim}.pkl', 'MOPSO-CD', exp_name) for exp_name in experiment_names]
if insert_nsga2:
  file_tuple_nsga2 = [(f'NSGA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSGA-II', exp_name) for exp_name in experiment_names]
if insert_nspso:
    file_tuple_nspso = [(f'NSPSO_{exp_name}_{objective_dim}_{position_dim}.pkl', 'NSPSO', exp_name) for exp_name in experiment_names]
if insert_spea2:
    file_tuple_spea2 = [(f'SPEA2_{exp_name}_{objective_dim}_{position_dim}.pkl', 'SPEA2', exp_name) for exp_name in experiment_names]


file_tuple = file_tuple_mesh + file_tuple_cmopso + file_tuple_maco + file_tuple_mopso_cd + file_tuple_nsga2 + file_tuple_nspso + file_tuple_spea2

# Take the results
results_mesh_igd = []
results_cmopso_igd = []
results_maco_igd = []
results_mopso_cd_igd = []
results_nsga2_igd = []
results_nspso_igd = []
results_spea2_igd = []

# Load the results from the files
for file_name, _, _ in file_tuple_mesh:
	with open(results_folder + file_name, "rb") as f:
		results_mesh_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_cmopso:
    with open(results_folder + file_name, "rb") as f:
        results_cmopso_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_maco:
    with open(results_folder + file_name, "rb") as f:
        results_maco_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_mopso_cd:
    with open(results_folder + file_name, "rb") as f:
        results_mopso_cd_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_nsga2:
	with open(results_folder + file_name, "rb") as f:
		results_nsga2_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_nspso:
    with open(results_folder + file_name, "rb") as f:
        results_nspso_igd.append(pickle.load(f))
for file_name, _, _ in file_tuple_spea2:
    with open(results_folder + file_name, "rb") as f:
        results_spea2_igd.append(pickle.load(f))

results = results_mesh_igd + results_cmopso_igd + results_maco_igd + results_mopso_cd_igd + results_nsga2_igd + results_nspso_igd + results_spea2_igd

# Create the pandas DataFrame columns
columns = [alg_name for (_, alg_name, _) in file_tuple]

igd_values = [[indicator(results[i][j+1]["F"]) for j in range(len(results[i])-1)] for i in range(len(results))]

df_igd = pd.DataFrame(list(zip(*igd_values)), columns=columns)
autorank_report_igd = autorank(df_igd, order='ascending', alpha=alpha_igd)

In [ ]:
autorank_igd = autorank_report_igd[0]
CD_igd = autorank_report_igd[2]
# If the differences between the mean ranks of two algorithms is greater than the critical distance, they are statistically different
print('Critical Distance (Inverse Generational Distance): ', CD_igd)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
	display(autorank_igd.sort_values(by=['meanrank'], ascending=True))